# Занятие 4, демо 1. Из прямой интерполяции не следует прямая траектория

При обучении мы берём пару $(x_0,x_1)$ и соединяем её **отрезком**:

$$
x_\tau=(1-\tau)x_0+\tau x_1,\qquad u_\tau=x_1-x_0
$$

Таргет постоянен вдоль всего отрезка. Отсюда легко сделать вывод, что и
генерация пойдёт по прямой, а значит хватит одного шага солвера.

Вывод не следует. Проверим числом.

In [ ]:
"""Общая база демо занятий 4 и 5: данные, сеть, метрика, солверы.

Здесь намеренно нет ни одной конструкции пути. На занятии 4 время идёт от шума
к данным, на занятии 5 - наоборот, и если держать обе ориентации в одном файле,
одно и то же имя начинает означать разное. Поэтому пути лежат по отдельности:
flow_cfm.py - занятие 4, flow_vp.py - занятие 5.

Направление интегрирования солверы получают аргументами t_from и t_to, а не
берут из умолчания: так его видно в месте вызова.

Датасет, сеть и расписание обучения те же, что в ДЗ-2, - иначе сравнение
занятий между собой перестало бы быть корректным.
"""
import math

import torch
from torch import nn

# --------------------------------------------------------------------------
# Данные: восемь гауссиан на окружности
# --------------------------------------------------------------------------

N_MODES = 8
RING_RADIUS = 2.0
MODE_STD = 0.15


def mode_centers(dtype=torch.float32) -> torch.Tensor:
    """Центры восьми компонент, форма [8, 2]."""
    k = torch.arange(N_MODES, dtype=dtype)
    angle = math.pi * k / 4
    return RING_RADIUS * torch.stack([torch.cos(angle), torch.sin(angle)], dim=-1)


def sample_data(n: int, generator: torch.Generator,
                dtype=torch.float32) -> torch.Tensor:
    """n точек из смеси восьми гауссиан, форма [n, 2]."""
    centers = mode_centers(dtype=dtype)
    which = torch.randint(N_MODES, (n,), generator=generator)
    noise = torch.randn(n, 2, generator=generator, dtype=dtype)
    return centers[which] + MODE_STD * noise


# --------------------------------------------------------------------------
# Сеть
# --------------------------------------------------------------------------

class VelocityNet(nn.Module):
    """Маленькая сеть: вход (x, tau) -> вектор в R^2. 4546 параметров."""

    def __init__(self, width: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, width),
            nn.SiLU(),
            nn.Linear(width, width),
            nn.SiLU(),
            nn.Linear(width, 2),
        )

    def forward(self, x: torch.Tensor, tau: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([x, tau], dim=-1))


def make_model(seed: int = 0, width: int = 64) -> VelocityNet:
    """Сеть с воспроизводимой инициализацией."""
    state = torch.get_rng_state()
    try:
        torch.manual_seed(seed)
        model = VelocityNet(width=width)
    finally:
        torch.set_rng_state(state)
    return model


# --------------------------------------------------------------------------
# Метрика: energy distance
# --------------------------------------------------------------------------

def energy_distance(x: torch.Tensor, y: torch.Tensor) -> float:
    """Оценка energy distance между двумя выборками. Считается в float64."""
    x = x.detach().to(torch.float64)
    y = y.detach().to(torch.float64)
    n, m = x.shape[0], y.shape[0]
    cross = torch.cdist(x, y).sum() / (n * m)
    inner_x = torch.cdist(x, x).sum() / (n * n)
    inner_y = torch.cdist(y, y).sum() / (m * m)
    return float(2 * cross - inner_x - inner_y)


# --------------------------------------------------------------------------
# Солверы и счётчик вызовов
# --------------------------------------------------------------------------

class CountingField:
    """Обёртка над полем, считающая вызовы."""

    def __init__(self, field):
        self.field = field
        self.calls = 0

    def __call__(self, x: torch.Tensor, tau: torch.Tensor) -> torch.Tensor:
        self.calls += 1
        return self.field(x, tau)


def euler_path(field, x: torch.Tensor, t_from: float, t_to: float,
               steps: int) -> torch.Tensor:
    """Явный Эйлер из t_from в t_to. Один шаг - один вызов поля.

    Возвращает все точки, форма [steps + 1, n, d].
    """
    if not isinstance(steps, int) or isinstance(steps, bool) or steps <= 0:
        raise ValueError("steps должно быть положительным int")

    h = (t_to - t_from) / steps
    ones = torch.ones(x.shape[0], 1, dtype=x.dtype, device=x.device)
    points = [x.clone()]

    with torch.no_grad():
        for m in range(steps):
            x = x + h * field(x, ones * (t_from + m * h))
            points.append(x.clone())

    return torch.stack(points)


def rk4_path(field, x: torch.Tensor, t_from: float, t_to: float,
             steps: int) -> torch.Tensor:
    """Классический RK4 из t_from в t_to. Один шаг - четыре вызова поля.

    Возвращает точки на границах макрошагов, форма [steps + 1, n, d].
    """
    if not isinstance(steps, int) or isinstance(steps, bool) or steps <= 0:
        raise ValueError("steps должно быть положительным int")

    h = (t_to - t_from) / steps
    ones = torch.ones(x.shape[0], 1, dtype=x.dtype, device=x.device)
    points = [x.clone()]

    with torch.no_grad():
        for m in range(steps):
            t = t_from + m * h
            k1 = field(x, ones * t)
            k2 = field(x + 0.5 * h * k1, ones * (t + 0.5 * h))
            k3 = field(x + 0.5 * h * k2, ones * (t + 0.5 * h))
            k4 = field(x + h * k3, ones * (t + h))
            x = x + (h / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)
            points.append(x.clone())

    return torch.stack(points)


def euler_solve(field, x: torch.Tensor, t_from: float, t_to: float,
                nfe: int) -> torch.Tensor:
    """Эйлер с бюджетом nfe вызовов поля: это ровно nfe шагов."""
    return euler_path(field, x, t_from, t_to, nfe)[-1]


def rk4_solve(field, x: torch.Tensor, t_from: float, t_to: float,
              nfe: int) -> torch.Tensor:
    """RK4 с бюджетом nfe вызовов поля: это nfe // 4 макрошагов."""
    if not isinstance(nfe, int) or isinstance(nfe, bool) or nfe <= 0:
        raise ValueError("nfe должно быть положительным int")
    if nfe % 4:
        raise ValueError("для RK4 бюджет вызовов должен делиться на 4")
    return rk4_path(field, x, t_from, t_to, nfe // 4)[-1]


# --------------------------------------------------------------------------
# Обучающий драйвер
# --------------------------------------------------------------------------

TRAIN_CONFIG = {
    "batch_size": 512,
    "steps": 12000,
    "lr": 2e-3,
    "data_seed": 1234,
    "noise_seed": 5678,
    "model_seed": 0,
    "report_every": 2000,
}


# --------------------------------------------------------------------------
# Замороженные веса: кладём вместе с тем, чем они являются
# --------------------------------------------------------------------------

def save_checkpoint(path, model, objective: str, cfg, history) -> None:
    """Сохраняет веса вместе с конструкцией, конфигом и достигнутыми потерями."""
    torch.save({"state_dict": model.state_dict(),
                "objective": objective,
                "config": dict(cfg),
                "final_loss": sum(history[-500:]) / 500}, path)


def load_checkpoint(path, objective: str):
    """Загружает веса и проверяет, что это обещанная конструкция.

    Демо занятий 4 и 5 стоят на утверждении «одна архитектура, один бюджет
    обучения». Утверждение, которое нельзя проверить на месте, - это дыра:
    файл легко перепутать, и ошибка будет молчаливой. Поэтому конструкция
    лежит внутри файла и сверяется при загрузке.
    """
    blob = torch.load(path, map_location="cpu", weights_only=False)
    if blob["objective"] != objective:
        raise ValueError(f"{path}: обучено на '{blob['objective']}', "
                         f"ожидалось '{objective}'")
    model = make_model(seed=blob["config"]["model_seed"])
    model.load_state_dict(blob["state_dict"])
    model.eval()
    return model, blob


"""Занятие 4: прямая интерполяция и conditional flow matching.

Ориентация занятия 4: `noise` - источник, `data` - цель, tau идёт от 0 к 1,
генерация вперёд. Имя x_0 здесь не используется намеренно: на занятии 5 оно
означает ровно противоположное.

Требует flow_common.py.
"""
import torch



def straight_interpolation(noise, data, tau):
    """Прямой отрезок между источником и целью."""
    return (1.0 - tau) * noise + tau * data


def cfm_target(noise, data):
    """Производная вдоль отрезка: она постоянна и равна разности концов."""
    return data - noise


def cfm_loss(model, noise, data, tau):
    """Conditional flow matching на прямой интерполяции."""
    prediction = model(straight_interpolation(noise, data, tau), tau)
    return ((prediction - cfm_target(noise, data)) ** 2).sum(dim=-1).mean()


def train_cfm(cfg=None, verbose=True):
    """Обучение поля на прямой интерполяции. Возвращает (model, история)."""
    cfg = dict(TRAIN_CONFIG if cfg is None else cfg)
    model = make_model(seed=cfg["model_seed"])
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    data_gen = torch.Generator().manual_seed(cfg["data_seed"])
    noise_gen = torch.Generator().manual_seed(cfg["noise_seed"])
    batch = cfg["batch_size"]
    history = []

    for step in range(1, cfg["steps"] + 1):
        data = sample_data(batch, data_gen)
        noise = torch.randn(batch, 2, generator=noise_gen)
        tau = torch.rand(batch, 1, generator=noise_gen)

        loss = cfm_loss(model, noise, data, tau)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        history.append(loss.detach().item())

        if verbose and step % cfg["report_every"] == 0:
            window = history[-cfg["report_every"]:]
            print(f"  шаг {step:6d}   потери {sum(window) / len(window):.4f}")

    return model, history


def learned_field(model):
    """Поле ОДУ занятия 4: предсказание сети как есть, без множителей."""
    def field(x, tau):
        return model(x, tau)
    return field


def oracle_field(x, tau):
    """Точное population-optimal поле E[data - noise | x_tau = x].

    Нейросети здесь нет: для нашей смеси гауссиан условное среднее выписывается
    аналитически. Пусть a = 1 - tau, b = tau, s = MODE_STD, D = a^2 + b^2 s^2.
    Тогда веса компонент пропорциональны exp(-||x - b mu_k||^2 / 2D), а внутри
    компоненты условное среднее равно mu_k + (b s^2 - a) / D * (x - b mu_k).

    Нужно, чтобы отделить свойство конструкции от ошибки обучения: если кривые
    траектории видны и здесь, дело не в том, что сеть маленькая или недоучена.
    """
    a = 1.0 - tau                                          # [n, 1]
    b = tau                                                # [n, 1]
    mu = mode_centers(dtype=x.dtype)                       # [8, 2]
    var = a * a + b * b * MODE_STD ** 2                    # [n, 1]
    shift = x.unsqueeze(1) - b.unsqueeze(1) * mu           # [n, 8, 2]
    weights = torch.softmax(-(shift ** 2).sum(-1) / (2 * var), dim=-1)
    coefficient = ((b * MODE_STD ** 2 - a) / var).unsqueeze(1)
    per_mode = mu.unsqueeze(0) + coefficient * shift       # [n, 8, 2]
    return (weights.unsqueeze(-1) * per_mode).sum(dim=1)


def path_ratio(points):
    """Длина пути, делённая на длину хорды. Единица - прямая."""
    length = (points[1:] - points[:-1]).norm(dim=-1).sum(dim=0)
    chord = (points[-1] - points[0]).norm(dim=-1)
    return length / chord


def cfm_sample(model, noise, nfe):
    """Генерация полем занятия 4: Эйлер из tau=0 в tau=1."""
    return euler_solve(learned_field(model), noise, 0.0, 1.0, nfe)

## Обучаем поле

Ориентация занятия 4: источник - шум, цель - данные, время идёт от нуля к
единице. Датасет и сеть те же, что на занятии 5: восемь гауссиан на окружности,
сеть на 4.5 тысячи параметров. Обучение занимает около восьми секунд.

Рядом с потерями сети печатаем **неустранимую часть**: потери точного
population-оптимального поля на тех же данных. Условная дисперсия таргета
велика, и оптимальные потери вовсе не стремятся к нулю - без этой второй
строки число 3.8 не говорит ни о чём.

In [ ]:
model, _ = train_cfm(verbose=False)
model.eval()

g = torch.Generator().manual_seed(321)
data = sample_data(20_000, g)
noise = torch.randn(20_000, 2, generator=g)
tau = torch.rand(20_000, 1, generator=g)
x = straight_interpolation(noise, data, tau)
u = cfm_target(noise, data)

with torch.no_grad():
    learned = ((model(x, tau) - u) ** 2).sum(-1).mean()
    best = ((oracle_field(x, tau) - u) ** 2).sum(-1).mean()

print(f"потери обученной сети:  {float(learned):.3f}")
print(f"неустранимая часть:     {float(best):.3f}")

## Мера непрямизны

Длина траектории, делённая на длину хорды: $L/\lVert x(1)-x(0)\rVert$. У отрезка
она равна единице по построению, у непрямого пути больше.

Берём 512 стартов и интегрируем RK4 - чтобы в измерении не оказалось ошибки
Эйлера. И считаем то же самое для **точного** поля: если кривизна видна и без
нейросети, объяснение «сеть маленькая и недоучена» отпадает.

In [ ]:
starts = torch.randn(512, 2, generator=torch.Generator().manual_seed(3))

print(f"{'поле':<24} {'медиана':>9} {'доля > 1.05':>13}")
for name, field in (("точное population-поле", oracle_field),
                    ("обученная сеть", learned_field(model))):
    ratio = path_ratio(rk4_path(field, starts, 0.0, 1.0, 100))
    print(f"{name:<24} {float(ratio.median()):>9.2f}"
          f" {float((ratio > 1.05).float().mean()):>13.2f}")

## Картинка

Слева обучающие отрезки шести пар, справа траектории обученного поля из тех же
шести стартов. Кружок - старт.

Метки финиша означают разное: слева звезда - это вытянутая цель $x_1$ конкретной
пары, справа квадрат - точка, куда пришло ОДУ. Совпадать они не обязаны, и в
этом всё различие между conditional path и marginal flow.

In [ ]:
import matplotlib.pyplot as plt

g = torch.Generator().manual_seed(3)
six = torch.randn(6, 2, generator=g)
targets = sample_data(6, g)

grid = torch.linspace(0.0, 1.0, 101).reshape(-1, 1, 1)
segments = straight_interpolation(six, targets, grid)
curves = rk4_path(learned_field(model), six, 0.0, 1.0, 100)

cloud = sample_data(2000, torch.Generator().manual_seed(9))
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)

for ax, points, mark, title in ((axes[0], segments, "*", "обучающий отрезок"),
                                (axes[1], curves, "s", "траектория поля")):
    ax.scatter(cloud[:, 0], cloud[:, 1], s=5, alpha=0.3, color="0.7")
    for i in range(points.shape[1]):
        ax.plot(points[:, i, 0], points[:, i, 1], linewidth=2.4)
        ax.scatter(points[0, i, 0], points[0, i, 1], s=45, marker="o",
                   color="0.15", zorder=3)
        ax.scatter(points[-1, i, 0], points[-1, i, 1], s=80, marker=mark,
                   color="0.15", zorder=3)
    ax.set_title(title, fontsize=15)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

## Почему так

Отрезок соединяет **одну конкретную пару** $(x_0,x_1)$. Обученное поле этой пары
не знает: через ту же точку в тот же момент проходят отрезки разных пар. При
квадратичной ошибке оптимум регрессии - условное среднее

$$
v_\tau(x)=\mathbb E\bigl[x_1-x_0 \mid x_\tau=x\bigr]
$$

Оно зависит от $x$ и $\tau$ и не обязано совпадать со скоростью хоть одной
отдельной пары. Поэтому его интегральные кривые бывают непрямыми, хотя каждый
обучающий отрезок прям.

Для нашего датасета видно даже, куда именно ведёт эта кривизна. При $\tau=0$
условие $x_\tau=x$ - это просто $x_0=x$, а $x_1$ от него не зависит:

$$
v_0(x)=\mathbb E[x_1]-x=-x
$$

потому что среднее данных - центр кольца. То есть поле сначала тянет любую
стартовую точку **к нулю**, а ближе к концу должно развернуть её к одной из
восьми мод. Изгиб записан в самой конструкции, а не взялся от недоученности - на
точном поле он такой же.

Отсюда практический вывод: из «интерполяция прямая» не следует «хватит одного
шага». Один из способов сделать траектории прямее - менять **coupling**, то есть
какие пары соединяются, сохранив ту же линейную интерполяцию. Это уже отдельная
конструкция, и здесь мы её не показывали.